[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zoranduric/sd-hex-pipeline/blob/main/notebooks/build_balkans.ipynb)

# sd-hex-pipeline — build a Balkans hex map on Colab

Downloads commercially usable GIS data (Copernicus DEM, ESA WorldCover, Natural Earth, HydroRIVERS, RESOLVE Ecoregions),
bakes it into an ~11 km pointy-top hex grid and writes `hexmap.sdhx` for Godot (`godot/hex_map_data.gd`).

CPU runtime is enough (a GPU does not speed this up). For `REGION = "world"`: pick a **High-RAM CPU** runtime, set `USE_DRIVE = True` (the build resumes per latitude band after a disconnect) and try `HEX_KM = 50` first. **Runtime → Run all**; the result zip downloads at the end. See `DATA_SOURCES.md` for attribution you must ship with your game.

## 1. Settings

In [ ]:
REGION = "balkans"      # balkans, finland, baltics, india, middle_east, world, belgrade (small test) — or set BBOX
BBOX = None             # e.g. "13,34.5,30,48.5"  (min_lon,min_lat,max_lon,max_lat)
HEX_KM = 11.0           # 11 ≈ 2× Supreme Ruler density, 9.2 ≈ 3×, 8 ≈ 4×, 5.5 ≈ 8×
USE_DRIVE = False       # True = cache downloads + outputs in Google Drive (asks for permission); use for world
WORKERS = 16            # parallel downloads; 32 on High-RAM for world

## 2. Install
If the repo is **private**, add a Colab secret `GH_TOKEN` (key icon in the left sidebar) containing a
fine-grained GitHub token with *Contents: read-only* on `zoranduric/sd-hex-pipeline`, and allow notebook access.

In [ ]:
REPO = "github.com/zoranduric/sd-hex-pipeline"
try:
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
except Exception:
    _tok = None
_url = f"git+https://x-access-token:{_tok}@{REPO}" if _tok else f"git+https://{REPO}"
_r = !pip -q install "{_url}" 2>&1
if _r and any("error" in l.lower() for l in _r):
    raise RuntimeError("Install failed (private repo? set the GH_TOKEN secret). " + "\n".join(l.replace(_tok or "\0", "***") for l in _r[-5:]))
del _tok, _url
print("installed")


## 3. Storage

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/sd-hex-pipeline"
else:
    BASE = "/content"
CACHE, OUT = f"{BASE}/cache", f"{BASE}/out"
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)
print(CACHE, OUT)

## 4. Build

In [ ]:
import shlex, subprocess, sys
cmd = ["sdhex", "build", *(["--bbox", BBOX] if BBOX else ["--region", REGION]),
       "--hex-km", str(HEX_KM), "--out", OUT, "--cache", CACHE, "--workers", str(WORKERS), "-v"]
print("$", shlex.join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
log_lines, RUN = [], None
for line in proc.stdout:
    print(line, end=""); log_lines.append(line)
    if line.startswith("sdhx "):
        RUN = os.path.dirname(line.split(None, 1)[1].strip())
if proc.wait() != 0 or RUN is None:
    raise RuntimeError(f"Build failed (exit {proc.returncode}). The real error is in the log above, "
                       "usually the last 'Traceback' lines. Paste them into an issue / chat.")
print("\nOutput:", RUN)


## 5. Inspect

In [ ]:
import glob, json, os
from IPython.display import Image, display

run = globals().get("RUN")  # set by the build cell
if not run:
    runs = [p for p in glob.glob(f"{OUT}/*") if os.path.isfile(f"{p}/hexmap.meta.json")]
    if not runs:
        raise FileNotFoundError(f"No build output in {OUT}. Run the build cell and check its log for the error.")
    run = sorted(runs, key=os.path.getmtime)[-1]

with open(f"{run}/hexmap.meta.json") as f:
    meta = json.load(f)
print(run)
print(json.dumps(meta.get("stats", meta.get("crop")), indent=1)[:1500])
for img in ("preview.png", "preview_biomes.png"):
    if os.path.isfile(f"{run}/{img}"):
        display(Image(f"{run}/{img}", width=900))


## 6. Download zip (skipped when using Drive)

In [ ]:
import shutil
if not USE_DRIVE:
    from google.colab import files
    zip_path = shutil.make_archive(f"/content/{os.path.basename(run)}", "zip", run)
    files.download(zip_path)
else:
    print("Saved to Drive:", run)
